# Train Face to Hair Mapper

Build a lightweight frequency-based face-to-hair mapper from paired CelebA face cues and the reviewed hairstyle asset bank.

In [1]:
import json
from pathlib import Path
import sys

import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

PROJECT_ROOT = find_project_root()
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

from systems.static_auto_tryon.auto_app.ml.face_to_hair_mapper import (
    build_face_to_hair_training_records,
    build_mapper_payload,
    evaluate_mapper_predictions,
    save_mapper_payload,
    split_face_to_hair_records,
    write_mapper_records,
)

PROJECT_ROOT

WindowsPath('.')

In [2]:
ASSET_BANK_PATH = PROJECT_ROOT / 'backend' / 'data' / 'processed' / 'celeba_hair_rich_assets' / 'reviewed' / 'kept_assets_labeled_enriched.jsonl'
DATASET_ROOT = PROJECT_ROOT / 'backend' / 'data' / 'datasets' / 'face_to_hair_mapper_light'
TRAIN_MANIFEST = DATASET_ROOT / 'train.jsonl'
VAL_MANIFEST = DATASET_ROOT / 'val.jsonl'
SUMMARY_PATH = DATASET_ROOT / 'summary.json'
CHECKPOINT_PATH = PROJECT_ROOT / 'backend' / 'models' / 'face_to_hair_mapper_light' / 'mapper_config.json'

TRAIN_RATIO = 0.8
SEED = 42

ASSET_BANK_PATH

WindowsPath('./backend/data/processed/celeba_hair_rich_assets/reviewed/kept_assets_labeled_enriched.jsonl')

In [3]:
records = build_face_to_hair_training_records(ASSET_BANK_PATH)
train_records, val_records = split_face_to_hair_records(records, train_ratio=TRAIN_RATIO, seed=SEED)

summary = {
    'record_count': len(records),
    'train_records': len(train_records),
    'val_records': len(val_records),
    'face_field_counts': {
        field: pd.Series([row['face_labels'][field] for row in records]).value_counts().to_dict()
        for field in ('gender', 'face_fullness', 'cheekbones', 'hairline')
    },
    'hair_field_counts': {
        field: pd.Series([row['hair_labels'][field] for row in records]).value_counts().to_dict()
        for field in ('length', 'curl', 'style_family')
    },
}

DATASET_ROOT.mkdir(parents=True, exist_ok=True)
write_mapper_records(train_records, TRAIN_MANIFEST)
write_mapper_records(val_records, VAL_MANIFEST)
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('Total paired mapper records:', len(records))
print('Train records:', len(train_records))
print('Val records:', len(val_records))
pd.Series({
    'train_records': len(train_records),
    'val_records': len(val_records),
    'style_family_classes': len(summary['hair_field_counts']['style_family']),
})

Total paired mapper records: 72
Train records: 57
Val records: 15


train_records           57
val_records             15
style_family_classes     7
dtype: int64

In [4]:
mapper_payload = build_mapper_payload(train_records)
save_mapper_payload(mapper_payload, CHECKPOINT_PATH)

metrics = evaluate_mapper_predictions(val_records, mapper_payload)
metric_summary = {
    'exact_match_accuracy': metrics['exact_match_accuracy'],
    **metrics['field_accuracy'],
}

print(f'Saved mapper config to {CHECKPOINT_PATH}')
pd.Series(metric_summary)

Saved mapper config to backend/checkpoints\face_to_hair_mapper_light\mapper_config.json


exact_match_accuracy    0.533333
length                  0.933333
curl                    0.800000
style_family            0.533333
dtype: float64

In [5]:
prediction_df = pd.DataFrame(metrics['prediction_rows'])
prediction_df.head(10)

,asset_id,source_id,true_length,pred_length,correct_length,true_curl,pred_curl,correct_curl,true_style_family,pred_style_family,correct_style_family
0,celeba_hair_000120,000211.jpg,long,long,True,wavy,wavy,True,long_layered,long_layered,True
1,celeba_hair_000246,000426.jpg,long,long,True,wavy,wavy,True,long_layered,long_layered,True
2,celeba_hair_000213,000359.jpg,long,long,True,straight,wavy,False,bob,long_layered,False
3,celeba_hair_000014,000022.jpg,long,long,True,wavy,wavy,True,long_layered,long_layered,True
4,celeba_hair_000003,000007.jpg,short,short,True,straight,straight,True,pompadour,pompadour,True
5,celeba_hair_000009,000015.jpg,short,short,True,straight,straight,True,pompadour,pompadour,True
6,celeba_hair_000185,000318.jpg,short,short,True,straight,straight,True,pompadour,pompadour,True
7,celeba_hair_000029,000043.jpg,long,long,True,wavy,wavy,True,center_part,long_layered,False
8,celeba_hair_000037,000055.jpg,short,short,True,wavy,straight,False,side_part,pompadour,False
9,celeba_hair_000062,000096.jpg,medium,long,False,wavy,wavy,True,bob,long_layered,False
